  <a href="https://colab.research.google.com/github/marcpalo1999/MIA_sanidad/blob/main/2_3_OLD_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Aprendizaje Supervisado para Clasificación Médica: Predicción de Enfermedad Coronaria



 ## Objetivos

 En este script construiremos modelos de Machine Learning para clasificación médica usando el dataset de enfermedades cardíacas (nivel de estenosis coronaria) para predecir qué pacientes necesitan cateterismo cardíaco y cuales se lo pueden ahorrar.

In [ ]:
import os
if not os.path.exists('/content/MIA_sanidad'):
    !git clone https://github.com/marcpalo1999/MIA_sanidad.git
os.chdir('/content/MIA_sanidad')
os.getcwd()

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, classification_report, roc_curve, auc,
                             precision_recall_curve, average_precision_score)

# Modelos de clasificación
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone



 ---

 # SECCIÓN 0: Cargar los datos y dividir entre diseño y validación

In [ ]:
# Cargar el dataset
df = pd.read_csv("./data/heart_disease_dataset_con_nulos.csv")

print(f"Dataset cargado: {df.shape[0]} pacientes, {df.shape[1]} variables")
df.head()


 # SECCIÓN 0.1: Resumen Descripción del Problema Clínico

 Para un entendimiento completo del problema, leed el archivo markdown adjunto titulado `2_2_ML_Explicación_Datos.md`.



 **Contexto del problema:**



 La secuencia diagnóstica actual para enfermedad coronaria es:

 1. Evaluación clínica (edad, síntomas, factores de riesgo)

 2. ECG en reposo

 3. Prueba de esfuerzo con ECG

      - Si es ambigua → Gammagrafía cardíaca (mínimamente invasiva), para descartar
         
         - si la gammagrafia es positiva vamos a cateterismo terapeutico
         - si es gammagrafia es negativa dejamos en seguimiento/profilaxis 

      - Si positiva → Cateterismo, primeramente diagnostico (INVASIVO, riesgos, coste alto ~3000€)



 **Problema clínico:**

 - Muchos pacientes se someten a cateterismo innecesariamente

 - El cateterismo tiene riesgos (2% complicaciones: hematomas, nefropatía, etc.)

 - Alto coste económico



 **Solución propuesta con ML:**

 Predecir el resultado del cateterismo usando solo las pruebas no invasivas previas,

 para reducir cateterismos innecesarios manteniendo alta sensibilidad (no perder casos reales de enfermedad).



 **Dos estrategias de modelado:**



 1. **Modelo Básico**: Solo consulta + ECG + prueba esfuerzo (aplicable a TODOS los pacientes)

    - Variables: edad, sexo, tipo dolor torácico, presión arterial, colesterol, ECG reposo, resultados prueba esfuerzo

    - Objetivo: Screening inicial para evitar cateterismos innecesarios



 2. **Modelo Completo**: Incluye gammagrafía (solo para casos equívocos)

    - Variables: todas las anteriores + resultado gammagrafía (thal)

    - Objetivo: Optimizar decisión cuando ya se realizó gammagrafía



 **Variable excluida por data leakage:**

 - `ca` (número de vasos obstruidos visualizados en fluoroscopia) se obtiene DURANTE el cateterismo

 - Usar `ca` para predecir `num` (resultado cateterismo) es circular, ya que ambas se miden simultáneamente

 - Incluirla sería como hacer trampa: predecir el resultado usando información del mismo procedimiento



 **Binarización del target:**

 - Variable original `num`: 0 (sin obstrucción coronatia), 1-4 (diferentes grados de obstruccion en diferentes cavidades)

 - Simplificamos a binario: 0 = no necesita cateterismo, 1 = necesita cateterismo

 - Razón clínica: la decisión es binaria (hacer/no hacer cateterismo), no gradual

 ---

 # SECCIÓN 1: Entendiendo el Problema desde ML



 **Tipo de problema:**

 - Supervised Learning: tenemos variable objetivo (target) para entrenar

 - Classification: la variable objetivo es categórica (enfermedad: sí/no)



 **Objetivo:** Dado un nuevo paciente con sus características clínicas, predecir si tiene

 enfermedad coronaria significativa que requiera cateterismo.

---

## Mapa del Pipeline: ¿Dónde estamos en cada momento?

Antes de meternos en el código, aquí tienes una visión global del camino que vamos a seguir. Úsalo como mapa de referencia cuando no sepas en qué punto del proceso estás:

```
DATOS BRUTOS
    │
    ▼
[SECCIÓN 2]  EDA ──────────────── Entender el dataset, missings, distribuciones
    │
    ▼
[SECCIÓN 3]  Identificar variables categóricas (encoding pendiente)
    │
    ▼
[SECCIÓN 4]  Definir estrategias de modelado
    │
    ▼
[SECCIÓN 5]  ════════ TRAIN / TEST SPLIT ════════  ← Punto de corte crítico
                 │                        │
              X_train                  X_test
                 │                        │
                 ▼                        │
[SECCIÓN 5.1] One-Hot Encoding (separado, luego alineado con reindex)
                 │                        │
                 ▼                        │
[SECCIÓN 6]  Imputación (mediana aprendida SOLO de train)
                 │                        │
                 ▼                        │
[SECCIÓN 7]  Escalado (media/std aprendidos SOLO de train)
                 │                        │
                 ▼                        │
[SECCIONES 8-12]  Baseline + entrenamiento + comparación (SOLO en train)
                 │                        │
                 ▼                        ▼
[SECCIÓN 13] ════════ EVALUACIÓN FINAL ════════  ← Test usado UNA SOLA VEZ
    │
    ▼
[SECCIÓN 14] ROC, Precision-Recall, calibración, threshold
    │
    ▼
[SECCIÓN 15] Feature importance
    │
    ▼
[SECCIÓN 16] Threshold óptimo (CV en train) → Modelo final → Producción
```

**Regla de oro:** Todo lo que "aprende" del dataset (medianas, medias, stds, thresholds) \
se aprende ÚNICAMENTE de los datos de train. El test set es siempre ciego.

 ---

 # SECCIÓN 2: Análisis Exploratorio Rápido

In [ ]:
# Información general del dataset
df.info()


In [ ]:
# Revisar valores nulos
missing_counts = df.isna().sum().sort_values(ascending=False)
print("Número de missing values por columna:")
print(missing_counts)

Como veis, tenemos valores nulos en variables predictoras (thal, slope) y tambien en variable target (num), así como en ca, pero que nos da igual porque ya hemos dicho que la eliminaremos.

## Eliminación de pacientes sin información de la variable target "num":

Fijaos que es diferente como tratamos los valores nulos NaN cuando estan en la variable target que cuando estan en los predictores. Si estan en los predictores podemos escoger que hacer con esos valores: si inputar, borrar el paciente, borrar la variable entera para todos los pacientes etc. (se explica más adelante). En canvio, si el valor nulo esta en la variable target, la observación/paciente no nos sirve de nada, ni para entrenar el modelo, ni para testearlo, ni para ninguna cosa. Tenemos que borrarlo.

In [ ]:
# Analizar variable objetivo 'num'
print("Distribución original de 'num':")
print(df['num'].value_counts().sort_index())

#Ved que si ponemos que no quite los NaN (dropna=False) nos los cuenta
print("Distribución original de 'num':")
print(df['num'].value_counts(dropna=False).sort_index())



In [ ]:
# Convertir a problema binario, ademas ya codificamos 0 como no enfermedad y 1 como enfermedad, no con strings, ya que si fuesen strings tendriamos que pasarlo a 0/1
df['target'] = (df['num'] > 0).astype(int)

#Veamos la distribución de la nueva variable target
print("\nDistribución binaria (target):")
print(df['target'].value_counts())
print(f"\nPrevalencia de enfermedad: {df['target'].mean()*100:.1f}%")

 **Observación:** El dataset está relativamente balanceado (~45% con enfermedad).

 Esto es favorable para el aprendizaje del modelo. Aunque no refleja la prevalencia real en screening poblacional de estenosis coronaria (sería mucho menor), el grupo de pacientes sobre los que se ha creado el dataset y tambien sobre el que se aplicará el modelo ya fueron seleccionados para cateterismo basándose en criterio clínico previo.

In [ ]:
df = df.dropna(subset=['num'])
print(f"\nDespués de eliminar pacientes sin 'num': {df.shape[0]} pacientes restantes.")

In [ ]:
# Estadísticas descriptivas
df.describe()


 **Observaciones:**

 - Variables con diferentes escalas (edad: 29-77, colesterol: 126-564)

 - Algunas variables son categóricas codificadas como números

 - Presencia de valores faltantes que requieren manejo

 ## Verificación de duplicados - Evitar data leakage



 **Problema:** Si el mismo paciente aparece múltiples veces en el dataset, podría

 terminar en AMBOS conjuntos (train y test), causando data leakage severo.



 **Consecuencia:** El modelo "vería" al paciente durante el entrenamiento y luego

 lo "evaluaría" en el test, inflando artificialmente las métricas.



 **Solución:** Verificar que cada fila representa un paciente único.

In [ ]:
# Verificar duplicados completos (filas idénticas)
duplicados_completos = df.duplicated().sum()

if duplicados_completos > 0:
    print(f"ALERTA: {duplicados_completos} filas duplicadas encontradas")
    df = df.drop_duplicates()
    print(f"Duplicados eliminados. Dataset: {df.shape[0]} pacientes")
else:
    print(f"OK: No hay duplicados ({df.shape[0]} pacientes únicos)")

# Verificar duplicados parciales basados en variables clave
columnas_clave = ['age', 'sex', 'trestbps', 'chol', 'thalach']
duplicados_parciales = df.duplicated(subset=columnas_clave).sum()

if duplicados_parciales > 0:
    print(f"ALERTA: {duplicados_parciales} posibles pacientes repetidos (verificar manualmente)")


### Eliminación de variables residuales que no queremos (ca y num)

Como la información de la variable target original num ya la tenemos codificada en una nueva llamada target, a partir de ahora solo trabajaremos con target. Num la tenemos que borrar para evitar situaciones indeseadas como ponerla sin querer dentro de las variables predictores y generar data leakage de target en predictores indirectamente.

Tambien tenemos que borrar la variable ca por la misma razón, porque es una variable que contiene información directa del proceso que queremos predecir y no la tendremos en el futuro.

Para eliminarlas usaremos el metodo dataframe.drop().

Este metodo se puede usar tanto para borrar columnas como filas (usando el combre como df.drop(columns=[ca,num], axis = 1)) o por filas, donde le indicamos el indice de la fila que queremos borrar df.drop([2,5], axis = 0)

Fijaos que axis = 0 indica fila y axis = 1 indica columnas, igual que en la notación matemática para matrices i,j.

In [ ]:
df = df.drop(columns=['ca', 'num'], axis=1)
print(f"\nDespués de eliminar variables 'ca' y 'num': {df.shape[1]} variables restantes.")


 ---

 # SECCIÓN 3: Preprocesamiento - Parte 1: Codificación de Variables Categóricas



 Las variables categóricas necesitan One-Hot Encoding para ser interpretadas correctamente por parte de los modelos de ML.

 Sin él, el modelo pensaría que cp=4 es "mayor" que cp=2, lo cual puede tener sentido clinico en algunos casos (variables categoricas ordinales), pero no en todos.

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
# Variables categóricas en el dataset
categorical_cols = ['cp', 'restecg', 'slope']
categorical_cols = [col for col in categorical_cols if col in df.columns]

print(f"Variables categóricas identificadas: {categorical_cols}")
for col in categorical_cols:
    print(f"  {col}: {df[col].nunique()} categorías - valores: {sorted(df[col].unique())}")


### One-hot encoding

Como podemos ver, los autores del dataset muy amablemente ya nos han codificado las variables categoricas ['cp', 'restecg', 'slope'] como numéricas. Entonces no tenemos que hacer el mapeo str -> int. Lo que una variable categorica de Ints puede ser un problema para un modelo de ML si realmente no es ordinal, ya que decirle al modelo que la variable tiene valores del 1 al 4, implicitamente le indica que la categoria 4 > 3 > 2 > 1. Para evitar este efecto indeseado (si fuese indeseado) se usa el one-hot encoding, que aplicaremos ahora.

**Lógica**: Convierte variables categóricas en columnas binarias (0 o 1) para que los algoritmos de ML puedan procesarlas numéricamente sin asumir un orden o jerarquía falsa entre categorías (ej: "rojo"=1, "azul"=2, "verde"=3 implicaría incorrectamente que verde > azul > rojo).

**Cuándo aplicarlo**: Úsalo con variables categóricas nominales (sin orden inherente) como colores, países, o tipos de producto, especialmente en algoritmos basados en distancias (KNN, SVM, redes neuronales) o lineales (regresión lineal/logística).

**Cuándo NO aplicarlo**: Evítalo con variables ordinales (que tienen orden natural como "bajo", "medio", "alto")

In [ ]:
# Las variables categóricas ya están identificadas en categorical_cols.
# La transformación One-Hot Encoding la aplicamos DESPUÉS del split train/test
# (Sección 5.1), siguiendo el mismo principio que con la imputación y el escalado:
# ninguna transformación se "aprende" usando datos de test.

print(f"Variables a codificar: {categorical_cols}")
print("La transformación get_dummies() se aplica en la Sección 5.1, tras el split.")


Explicación del One-Hot Encoding:

El concepto básico:

Imagina que tienes una columna "Tipo de Mascota" con tres valores posibles: "Perro", "Gato" y "Pájaro". El one-hot encoding transforma esta única columna en tres columnas separadas, una para cada tipo de mascota posible. Cada nueva columna responde a una pregunta simple de sí o no: "¿Es un perro?", "¿Es un gato?", "¿Es un pájaro?".

Cómo funciona la transformación:

Si una fila tenía "Perro" en la columna original, ahora tendrá un 1 en la columna "Es_Perro" y un 0 en las columnas "Es_Gato" y "Es_Pájaro". Si otra fila tenía "Gato", tendrá un 0 en "Es_Perro", un 1 en "Es_Gato" y un 0 en "Es_Pájaro". Es como un sistema de interruptores donde solo uno puede estar encendido a la vez.

 ---

 # SECCIÓN 4: Definición de Estrategias de Modelado



 Implementaremos DOS estrategias según disponibilidad de datos clínicos.

In [ ]:
# ESTRATEGIAS DE MODELADO
# Las listas exactas de features las definiremos en la Sección 5.1,
# una vez que tengamos las columnas exactas tras el One-Hot Encoding.
#
# La lógica es:
#   ESTRATEGIA 1 (Básico):   todas las variables EXCEPTO 'thal' (gammagrafía)
#   ESTRATEGIA 2 (Completo): TODAS las variables incluyendo 'thal'
#   Variable EXCLUIDA en ambas: 'ca' (data leakage, ya eliminada en Sección 2)

print("ESTRATEGIAS DE MODELADO:")
print("  Básica   → Sin gammagrafía 'thal' (aplicable a cualquier paciente)")
print("  Completa → Con gammagrafía 'thal' (solo cuando ya se ha realizado)")
print("\nListas exactas de features: ver Sección 5.1 (después del encoding y el split)")


**Observación:** Vemos que ahora correctamente la unica diferencia entre las variables del modelo básico y el completo es la variable thal, perteneciente a la gammagrafia de perfusión, y que en ninguna de las dos listas de variables aparece ninguna variable objetivo o similares (num, ca, target)

 ---

 # SECCIÓN 5: División Train/Test



 ## El problema fundamental del ML



 Si entrenamos y evaluamos en los mismos datos, el modelo memorizará en lugar de aprender.

 Como darle a un estudiante las preguntas del examen antes del examen.

 Cuando evaluamos la performance de nuestro modelo, lo que estamos haciendo es siempre intentar aproximar la performance en producción (en el mundo real). Como los datos del mundo real no los tenemos nunca (ya que aun estamos diseñando el modelo, no poniendolo en producción) lo que hacemos es buscar diferentes metodos de aproximar esto.

 El primero sería evaluar el modelo en los mismos datos que usamos para entrenarlo, en el train. Como ya hemos dicho, esta no sera una buena aproximación de la realidad, ya que el modelo se sabe mucho mejor los datos con los que ha sido entrenado de lo que se sabrá los de producción.

 La segunda opción es dividir los datos en train y test, entrenar y diseñar solo en train y usar los datos de test como datos ciegos, que el modelo no ha visto nunca, emulando que son datos de producción.

 Hay técnicas más sofisticadas al respecto como la validación cruzada para estimar la performance del modelo en la realidad, que veremos también más adelante.



 ## Solución: Train/Test Split



 - **Train set (80%):** Entrenar el modelo

 - **Test set (20%):** Completamente oculto, simula pacientes nuevos

 Usar el 80% de los datos para entrenar y el 20 para testear es una convención, se podria usar cualquier proporcion deseada, cuidando que el modelo tenga suficiente datos en train como para aprender a clasificar el problema.

 El test set se usa UNA SOLA VEZ al final para estimar performance real.

In [ ]:
# Preparar datos para el split
# Usamos df directamente (sin codificar todavía las categóricas)
X = df.drop(columns=['target'])
y = df['target']

print(f"Dimensiones X: {X.shape}")
print(f"Columnas (antes del encoding): {list(X.columns)}")

# Train/Test Split (80/20) con estratificación para mantener proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain: {X_train.shape[0]} pacientes ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} pacientes ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nVerificación de estratificación:")
print(f"  Enfermedad en Train:    {y_train.mean()*100:.1f}%")
print(f"  Enfermedad en Test:     {y_test.mean()*100:.1f}%")
print(f"  Enfermedad en Original: {y.mean()*100:.1f}%")


### Explicación del random_state y stratify:

¿Por qué random_state=42?

El random_state es como fijar una "semilla" para el generador de números aleatorios. Sin él, cada vez que ejecutes el código obtendrías una división diferente de train/test (diferentes pacientes en cada conjunto), lo que haría imposible reproducir tus resultados o comparar experimentos. Al fijar random_state=42 (o cualquier número), garantizas que siempre obtendrás exactamente la misma división de datos, permitiendo que tú o cualquier persona pueda replicar tus resultados exactos. El número 42 es una convención popular (referencia a "La Guía del Autoestopista Intergaláctico"), pero podrías usar cualquier número.

¿Por qué stratify=y?

La estratificación es crucial cuando tienes clases desbalanceadas. Imagina que en tu dataset original el 30% de pacientes tienen la enfermedad y el 70% están sanos. Sin estratificación, por puro azar podrías terminar con un conjunto de test donde solo el 20% tiene la enfermedad, o peor aún, donde el 45% la tiene. Esto distorsionaría completamente la evaluación de tu modelo.
Al usar stratify=y, le dices a scikit-learn: "mantén la misma proporción de enfermos/sanos en train y test que existe en el dataset original". Si tienes 30% de enfermos en total, tendrás aproximadamente 30% en train Y 30% en test. Esto es especialmente importante en problemas médicos donde la clase minoritaria (los enfermos) es la más importante de detectar correctamente.

---

# SECCIÓN 5.1: One-Hot Encoding (DESPUÉS del split)

Ahora sí aplicamos la transformación identificada en la Sección 3.

**¿Por qué aquí y no antes?** Con `get_dummies`, a diferencia de la imputación o el \
escalado, no se "aprende" ninguna estadística del dataset. Simplemente expande columnas \
según los valores únicos que encuentra. Sin embargo, el problema práctico es que **si \
una categoría solo aparece en test y no en train**, se generarían columnas distintas en \
cada set, rompiendo el modelo en producción.

La solución es aplicarlo por separado a train y test, y luego **alinear las columnas** \
con `reindex`. Si alguna categoría falta en test, se rellena con 0 (no apareció = no activa).

En este dataset el riesgo es mínimo (todos los valores de cp, restecg y slope aparecen \
en ambos sets), pero la práctica correcta es seguir siempre el pipeline consistente.

In [ ]:
# Aplicar One-Hot Encoding DESPUÉS del split (por separado en train y test)
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=categorical_cols, drop_first=True, dtype=int)

# Alinear columnas: si alguna categoría no aparece en test, se añade con valor 0
# Esto garantiza que train y test tienen exactamente las mismas columnas
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"Shape X_train tras encoding: {X_train.shape}")
print(f"Shape X_test  tras encoding: {X_test.shape}")
nuevas = [c for c in X_train.columns if any(cat in c for cat in categorical_cols) and '_' in c]
print(f"\nColumnas nuevas generadas: {nuevas}")


In [ ]:
# Definir features para cada estrategia ahora que tenemos las columnas exactas
all_features = list(X_train.columns)

# ESTRATEGIA 1: sin 'thal' (gammagrafía de perfusión)
basic_features    = [col for col in all_features if col != 'thal']
# ESTRATEGIA 2: con 'thal'
complete_features = all_features.copy()

print(f"ESTRATEGIA 1 — Modelo Básico:   {len(basic_features)} variables")
print(f"ESTRATEGIA 2 — Modelo Completo: {len(complete_features)} variables")
print(f"Variable adicional en Completo: {[c for c in complete_features if c not in basic_features]}")
print(f"\nLista básico: {basic_features}")


 ---

 # SECCIÓN 6: Preprocesamiento - Parte 2: Valores Faltantes (DESPUÉS del split para evitar data leakage)



 ## Por qué DESPUÉS del split



 Los modelos ML no pueden trabajar con valores NaN. Usaremos mediana para imputación (robusta a outliers).



 **Flujo incorrecto:**

 1. Calcular mediana con todos los datos (train + test)

 2. Imputar valores faltantes

 3. Hacer train/test split



 **Problema:** La mediana fue calculada usando información del test set, causando data leakage.



 **Flujo correcto:**

 1. Hacer train/test split primero

 2. Calcular mediana SOLO con datos de train

 3. Aplicar esa mediana a train

 4. Aplicar la MISMA mediana (del train) al test



 **Analogía clínica:** Es como calcular valores de referencia de un test diagnóstico.

 Los valores de referencia se calculan con la población de desarrollo, NO incluyendo

 la población de validación. Luego se aplican esos mismos valores a la validación.

In [ ]:
columns_with_nulls = X_train.columns[X_train.isnull().any()].tolist()

if len(columns_with_nulls) > 0:
    print(f"Variables con valores nulos: {columns_with_nulls}")
    print(f"\nValores nulos antes de imputación:")
    print(f"  Train: {X_train.isnull().sum().sum()}")
    print(f"  Test: {X_test.isnull().sum().sum()}")
    
    # Imputación con mediana calculada SOLO del train
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train)  # Aprende SOLO de train
    
    X_train = pd.DataFrame(
        imputer.transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )
    
    X_test = pd.DataFrame(
        imputer.transform(X_test),  # Usa medianas del train
        columns=X_test.columns,
        index=X_test.index
    )
    
    print(f"\nValores nulos después de imputación:")
    print(f"  Train: {X_train.isnull().sum().sum()}")
    print(f"  Test: {X_test.isnull().sum().sum()}")
else:
    print("No hay valores nulos en train ni test")


 ---

 # SECCIÓN 7: Escalado (DESPUÉS del split para evitar data leakage)



 ## Por qué escalar

 Las variables tienen escalas muy diferentes (edad: 29-77, colesterol: 126-564).

 Algunos algoritmos (Logistic Regression, KNN) son sensibles a estas diferencias, porque estan basados internamente en metricas de "distancia".



 ## StandardScaler

 Transforma a media=0, desviación estándar=1.



 ## Por qué DESPUÉS del split

 Si escalamos antes, calcularíamos estadísticas usando también el test set, lo cual sería data leakage.



 **Flujo correcto:**

 1. Split train/test

 2. Calcular media/std SOLO del train

 3. Aplicar transformación a ambos sets

In [ ]:
scaler = StandardScaler()

# Fit en train (aprende media y std)
X_train_scaled = scaler.fit_transform(X_train)
# Transform en test (usa estadísticas del train)
X_test_scaled = scaler.transform(X_test)

# Convertir a DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

X_train_scaled.describe()

**Observación:** Efectivamente nuestras variables continuas ahora tienen media 0 y std 1

 ---

 # SECCIÓN 8: Modelo Baseline (base)



 ## Por qué necesitamos un baseline



 Antes de construir modelos complejos, necesitamos un punto de referencia:
 el enfoque más simple posible.



 **DummyClassifier** siempre predice la clase más frecuente.

 Es nuestro "mínimo a superar".

In [ ]:
dummy_clf = DummyClassifier(strategy='most_frequent', random_state=42)
dummy_clf.fit(X_train_scaled, y_train)

dummy_train_pred = dummy_clf.predict(X_train_scaled)
dummy_test_pred = dummy_clf.predict(X_test_scaled)

dummy_train_acc = accuracy_score(y_train, dummy_train_pred)
dummy_test_acc = accuracy_score(y_test, dummy_test_pred)

most_frequent_class = dummy_clf.classes_[np.argmax(dummy_clf.class_prior_)]

print(f"BASELINE MODEL (DummyClassifier)")
print(f"  Train Accuracy: {dummy_train_acc*100:.2f}%")
print(f"  Test Accuracy: {dummy_test_acc*100:.2f}%")
print(f"  Predicción: siempre {'Enfermedad' if most_frequent_class == 1 else 'Sin Enfermedad'}")
print(f"\nCualquier modelo ML debe superar {dummy_test_acc*100:.1f}% para ser útil")


In [ ]:
dummy_test_pred

 ---

 # SECCIÓN 9: Entendiendo la Evaluación de Performance



 ## Tres formas de evaluar un modelo:

 En el fondo siempre que evaluamos un modelo la pregunta que nos hacemos es: Como de bien lo hará nuestro modelo en "produccion", es decir, cuando lo estemos usando en el departamento que le toque? Como no lo sabemos (porque aun no lo hemos puesto en producción), lo que intentamos es estimar lo mas precisamente posible como lo hará, y tres maneras de estimar este error en producción (aunque unas mejores que otras) son las siguientes:

 **A) Train Accuracy:** Detectar underfitting (modelo demasiado simple)



 **B) Test Accuracy:** Performance real (usar SOLO al final, UNA VEZ)



 **C) Cross-Validation:** Método preferido durante desarrollo del modelo



 Durante desarrollo usamos SOLO los datos de train para no "contaminar" el test set.

 El mismo ejercicio que estamos haciendo para la metrica de accuracy lo podriamos hacer para cualquier otra metrica de performance, somo sensitividad, especificidad etc.

In [ ]:
# Demostración con Logistic Regression
demo_model = LogisticRegression(random_state=42, max_iter=1000)
demo_model.fit(X_train_scaled, y_train)

# A) Train Accuracy
train_pred = demo_model.predict(X_train_scaled)
train_acc = accuracy_score(y_train, train_pred)

# B) Test Accuracy
test_pred = demo_model.predict(X_test_scaled)
test_acc = accuracy_score(y_test, test_pred)

# C) Cross-Validation
cv_scores = cross_val_score(demo_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("TRES FORMAS DE EVALUAR UN MODELO:\n")
print(f"A) TRAIN ACCURACY: {train_acc*100:.2f}%")
print("   El modelo 've' estos datos durante entrenamiento")
print("   Score alto NO garantiza funcionar en pacientes nuevos\n")

print(f"B) TEST ACCURACY: {test_acc*100:.2f}%")
print("   El modelo NUNCA vio estos datos")
print("   Estima performance real")
print("   Solo hacer esto UNA VEZ al final\n")

print(f"C) CROSS-VALIDATION (5-Fold):")
print(f"   Scores por fold: {[f'{s*100:.1f}%' for s in cv_scores]}")
print(f"   Media: {cv_scores.mean()*100:.2f}% (± {cv_scores.std()*100:.2f}%)")
print("   Usa solo training data (divide en 5 folds los datos de train)")
print("   Da múltiples estimaciones (más robusto)")
print("   Test set permanece intacto")


---

# SECCIÓN 9.1: Bias-Variance Tradeoff — El dilema central del ML

Antes de entrenar los cuatro modelos, necesitamos entender el concepto más importante de todo el aprendizaje supervisado: el **trade-off entre bias y varianza**.

## ¿Qué es el bias (sesgo)?

El bias es el error que comete un modelo por ser **demasiado simple** para capturar la complejidad real del problema. Un modelo con mucho bias hace suposiciones demasiado rígidas y "no aprende suficiente" de los datos.

**Ejemplo clínico:** Imagina predecir enfermedad coronaria usando solo una regla: "si edad > 55, enfermedad = sí". Muchos pacientes jóvenes enfermos quedarían sin detectar. Eso es alto bias — el modelo no tiene capacidad de representar la realidad.

**En ML:** Logistic Regression tiene más bias que Random Forest porque asume que la frontera de decisión entre sanos y enfermos es una **línea recta** en el espacio de features. Si la realidad es más compleja (curvilínea, con interacciones entre variables), el modelo nunca lo captará, por más datos que le des.

## ¿Qué es la varianza?

La varianza es el error que comete un modelo por ser **demasiado complejo**: aprende no solo el patrón real sino también el "ruido" particular de los datos de entrenamiento. Funciona perfectamente en train pero falla en datos nuevos.

**Ejemplo clínico:** Un Decision Tree sin límite de profundidad puede crear reglas tan específicas como "si thalach = 142 Y oldpeak = 1.4 Y sexo = 0 → enfermedad". Memoriza a los 242 pacientes de train. Cuando llega un paciente nuevo con thalach = 143, el árbol no sabe qué hacer. Eso es alta varianza: aprendió de memoria casos concretos, no el patrón general.

Lo detectamos cuando **Train Accuracy >> CV Accuracy** (gap grande entre ambas).

## El trade-off

No podemos minimizar bias y varianza simultáneamente. Hacer el modelo más complejo reduce bias pero aumenta varianza, y viceversa. El objetivo es encontrar el equilibrio:

```
Error total = Bias² + Varianza + Ruido irreducible (no eliminable)
```

| Modelo | Bias | Varianza | Intuición |
|---|---|---|---|
| Logistic Regression | Alto | Bajo | Simple y estable, puede no capturar complejidad |
| Decision Tree (sin límite) | Bajo | **Muy alto** | Flexible pero tiende a memorizar |
| Decision Tree (max_depth=5) | Medio | Medio | Limitamos profundidad para controlar varianza |
| **Random Forest** | Bajo | **Muy bajo** | 100 árboles → su varianza individual se promedia |
| KNN (K=5) | Medio | Medio | K pequeño = más varianza; K grande = más bias |

## ¿Cómo lo vemos en práctica?

Cuando entrenamos los modelos a continuación, fíjate en la columna **Gap (Train - CV)**:
- **Gap grande (>10-15%):** alta varianza → el modelo sobreajusta (overfitting)
- **CV baja con gap pequeño:** puede ser underfitting (alto bias) — el modelo es demasiado simple

Random Forest suele ganar porque **promedia las predicciones de 100 árboles** (Bootstrap AGGregating = **bagging**): cada árbol individual tiene alta varianza, pero su media tiene varianza muy baja. Eso es exactamente lo que veremos.

 ---

 # SECCIÓN 10: ESTRATEGIA 1 - Modelo Básico (Sin Gammagrafía)



 Entrenamos modelos usando solo variables de consulta + prueba esfuerzo.

 Aplicable a TODOS los pacientes.

In [ ]:
# Preparar datos para Modelo Básico
X_train_basic = X_train_scaled[basic_features]
X_test_basic = X_test_scaled[basic_features]

print(f"ESTRATEGIA 1: MODELO BÁSICO")
print(f"Variables usadas: {len(basic_features)} de {len(complete_features)} totales")
print(f"Excluidas: thal (gammagrafía), ca (cateterismo)")
print(f"Train: {X_train_basic.shape}, Test: {X_test_basic.shape}")


### Modelo 1: Regresión Logística

**Supuesto clave:** Asume que la frontera de decisión entre clases es **lineal**: existe un hiperplano (línea recta en 2D, plano en 3D) que separa perfectamente sanos de enfermos en el espacio de features. Si la relación real es curvilínea o hay interacciones entre variables, la regresión logística nunca lo capturará.

**¿Por qué usarla entonces?**
1. Es muy **interpretable**: el coeficiente de cada variable dice directamente cuánto y en qué dirección contribuye al riesgo (similar a una regresión logística clásica epidemiológica)
2. **Entrenamiento rapidísimo** y numéricamente estable
3. Excelente **punto de referencia**: si un modelo más complejo no la supera claramente, igual de sencillo usar ésta

**Señal de bias alto:** Train y CV accuracy similares entre sí, pero ambos moderados (el modelo no puede aprender más aunque tenga más datos).

In [ ]:
# Modelo 1: Logistic Regression (Básico)
log_reg_basic = LogisticRegression(random_state=42, max_iter=1000)
log_reg_basic.fit(X_train_basic, y_train)

log_reg_basic_train = log_reg_basic.score(X_train_basic, y_train)
log_reg_basic_cv = cross_val_score(log_reg_basic, X_train_basic, y_train, cv=5, scoring='accuracy')

print("MODELO 1: LOGISTIC REGRESSION (Básico)")
print(f"Train: {log_reg_basic_train*100:.2f}% | CV: {log_reg_basic_cv.mean()*100:.2f}% (±{log_reg_basic_cv.std()*100:.2f}%) | Gap: {(log_reg_basic_train - log_reg_basic_cv.mean())*100:.2f}%")


### Modelo 2: Decision Tree

**Mecanismo:** Construye una serie de preguntas binarias ("¿thalach > 150?", "¿oldpeak > 2.3?") que forman un árbol. Cada hoja del árbol corresponde a una decisión final.

**Supuesto clave:** Ninguno sobre la distribución de los datos. Puede dividir el espacio de features de forma arbitrariamente compleja si se lo permitimos.

**El problema del overfitting:** Sin `max_depth`, el árbol crece hasta memorizar cada paciente de entrenamiento individualmente — alta varianza. Usamos `max_depth=5` para limitarlo. Este parámetro es un **dial de varianza**: más profundidad = más varianza; menos profundidad = más bias.

**Señal de varianza alta:** Train accuracy >> CV accuracy. Si pruebas a usar `max_depth=None` (sin límite), verás que Train llega al 100% pero CV cae. Eso es overfitting puro.

In [ ]:
# Modelo 2: Decision Tree (Básico)
tree_basic = DecisionTreeClassifier(random_state=42, max_depth=5)
tree_basic.fit(X_train_basic, y_train)

tree_basic_train = tree_basic.score(X_train_basic, y_train)
tree_basic_cv = cross_val_score(tree_basic, X_train_basic, y_train, cv=5, scoring='accuracy')

print("MODELO 2: DECISION TREE (Básico)")
print(f"Train: {tree_basic_train*100:.2f}% | CV: {tree_basic_cv.mean()*100:.2f}% (±{tree_basic_cv.std()*100:.2f}%) | Gap: {(tree_basic_train - tree_basic_cv.mean())*100:.2f}%")


### Modelo 3: Random Forest

**Mecanismo:** Entrena 100 árboles de decisión independientes. Cada árbol se entrena en una **muestra aleatoria con reemplazo** de los datos (bootstrap) y en cada división solo puede usar un **subconjunto aleatorio de features**. La predicción final es la votación mayoritaria de todos los árboles.

**¿Por qué funciona mejor que un árbol solo?** Cada árbol individual tiene alta varianza (sobreajusta a su muestra), pero al **promediar 100 votos** los errores individuales se cancelan. Si un árbol se equivoca porque vio una muestra rara, los otros 99 compensan. Esto reduce la varianza a casi cero sin aumentar el bias: es el **bagging** que vimos en la sección anterior, aplicado en la práctica.

**Resultado esperado:** Gap Train-CV mucho menor que en el Decision Tree individual, y CV accuracy superior. Si no ocurre así, sería una señal de que el problema tiene muy alta irreducibilidad (mucho ruido intrínseco).

In [ ]:
# Modelo 3: Random Forest (Básico)
rf_basic = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_basic.fit(X_train_basic, y_train)

rf_basic_train = rf_basic.score(X_train_basic, y_train)
rf_basic_cv = cross_val_score(rf_basic, X_train_basic, y_train, cv=5, scoring='accuracy')

print("MODELO 3: RANDOM FOREST (Básico)")
print(f"Train: {rf_basic_train*100:.2f}% | CV: {rf_basic_cv.mean()*100:.2f}% (±{rf_basic_cv.std()*100:.2f}%) | Gap: {(rf_basic_train - rf_basic_cv.mean())*100:.2f}%")


### Modelo 4: K-Nearest Neighbors (KNN)

**Mecanismo:** Para predecir si un nuevo paciente está enfermo, busca los K=5 pacientes más "cercanos" en el espacio de features por distancia euclidiana, y decide por votación mayoritaria entre ellos.

**Supuesto clave:** Pacientes con características clínicas similares (edad, colesterol, thalach parecidos...) deberían tener el mismo diagnóstico. La "similitud" se mide geométricamente como distancia en el espacio de features — por eso el **escalado (Sección 7) es absolutamente crítico aquí**: sin escalar, el colesterol (rango 126-564) dominaría completamente sobre el sexo (0-1), haciendo que "cercano" signifique "colesterol parecido" más que "perfil clínico similar".

**Limitaciones:**
- Lento en producción: debe calcular distancias a todos los pacientes de train
- Sensible a features irrelevantes que distorsionan las distancias
- Poca interpretabilidad: no hay una fórmula, solo "se parece a estos 5 pacientes"

In [ ]:
# Modelo 4: KNN (Básico)
knn_basic = KNeighborsClassifier(n_neighbors=5)
knn_basic.fit(X_train_basic, y_train)

knn_basic_train = knn_basic.score(X_train_basic, y_train)
knn_basic_cv = cross_val_score(knn_basic, X_train_basic, y_train, cv=5, scoring='accuracy')

print("MODELO 4: K-NEAREST NEIGHBORS (Básico)")
print(f"Train: {knn_basic_train*100:.2f}% | CV: {knn_basic_cv.mean()*100:.2f}% (±{knn_basic_cv.std()*100:.2f}%) | Gap: {(knn_basic_train - knn_basic_cv.mean())*100:.2f}%")


 ---

 # SECCIÓN 11: ESTRATEGIA 2 - Modelo Completo (Con Gammagrafía)



 Entrenamos modelos incluyendo resultado de gammagrafía.

 Solo aplicable a pacientes que ya tienen gammagrafía.

In [ ]:
print(f"ESTRATEGIA 2: MODELO COMPLETO")
print(f"Variables usadas: {len(complete_features)}")
print(f"Incluye: thal (gammagrafía)")
print(f"Excluye: ca (cateterismo - data leakage)")
print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")


In [ ]:
# Modelo 1: Logistic Regression (Completo)
log_reg_complete = LogisticRegression(random_state=42, max_iter=1000)
log_reg_complete.fit(X_train_scaled, y_train)

log_reg_complete_train = log_reg_complete.score(X_train_scaled, y_train)
log_reg_complete_cv = cross_val_score(log_reg_complete, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("MODELO 1: LOGISTIC REGRESSION (Completo)")
print(f"Train: {log_reg_complete_train*100:.2f}% | CV: {log_reg_complete_cv.mean()*100:.2f}% (±{log_reg_complete_cv.std()*100:.2f}%) | Gap: {(log_reg_complete_train - log_reg_complete_cv.mean())*100:.2f}%")


In [ ]:
# Modelo 2: Decision Tree (Completo)
tree_complete = DecisionTreeClassifier(random_state=42, max_depth=5)
tree_complete.fit(X_train_scaled, y_train)

tree_complete_train = tree_complete.score(X_train_scaled, y_train)
tree_complete_cv = cross_val_score(tree_complete, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("MODELO 2: DECISION TREE (Completo)")
print(f"Train: {tree_complete_train*100:.2f}% | CV: {tree_complete_cv.mean()*100:.2f}% (±{tree_complete_cv.std()*100:.2f}%) | Gap: {(tree_complete_train - tree_complete_cv.mean())*100:.2f}%")


In [ ]:
# Modelo 3: Random Forest (Completo)
rf_complete = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_complete.fit(X_train_scaled, y_train)

rf_complete_train = rf_complete.score(X_train_scaled, y_train)
rf_complete_cv = cross_val_score(rf_complete, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("MODELO 3: RANDOM FOREST (Completo)")
print(f"Train: {rf_complete_train*100:.2f}% | CV: {rf_complete_cv.mean()*100:.2f}% (±{rf_complete_cv.std()*100:.2f}%) | Gap: {(rf_complete_train - rf_complete_cv.mean())*100:.2f}%")


In [ ]:
# Modelo 4: KNN (Completo)
knn_complete = KNeighborsClassifier(n_neighbors=5)
knn_complete.fit(X_train_scaled, y_train)

knn_complete_train = knn_complete.score(X_train_scaled, y_train)
knn_complete_cv = cross_val_score(knn_complete, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("MODELO 4: K-NEAREST NEIGHBORS (Completo)")
print(f"Train: {knn_complete_train*100:.2f}% | CV: {knn_complete_cv.mean()*100:.2f}% (±{knn_complete_cv.std()*100:.2f}%) | Gap: {(knn_complete_train - knn_complete_cv.mean())*100:.2f}%")


 ---

 # SECCIÓN 12: Comparación de Estrategias

In [ ]:
# Tabla comparativa
comparison = pd.DataFrame({
    'Modelo': ['Baseline', 
               'LogReg (Básico)', 'Tree (Básico)', 'RF (Básico)', 'KNN (Básico)',
               'LogReg (Completo)', 'Tree (Completo)', 'RF (Completo)', 'KNN (Completo)'],
    'Estrategia': ['N/A',
                   'Básico', 'Básico', 'Básico', 'Básico',
                   'Completo', 'Completo', 'Completo', 'Completo'],
    'Train (%)': [dummy_train_acc*100,
                  log_reg_basic_train*100, tree_basic_train*100, rf_basic_train*100, knn_basic_train*100,
                  log_reg_complete_train*100, tree_complete_train*100, rf_complete_train*100, knn_complete_train*100],
    'CV (%)': [dummy_test_acc*100,
               log_reg_basic_cv.mean()*100, tree_basic_cv.mean()*100, rf_basic_cv.mean()*100, knn_basic_cv.mean()*100,
               log_reg_complete_cv.mean()*100, tree_complete_cv.mean()*100, rf_complete_cv.mean()*100, knn_complete_cv.mean()*100],
    'Gap (%)': [0,
                (log_reg_basic_train - log_reg_basic_cv.mean())*100,
                (tree_basic_train - tree_basic_cv.mean())*100,
                (rf_basic_train - rf_basic_cv.mean())*100,
                (knn_basic_train - knn_basic_cv.mean())*100,
                (log_reg_complete_train - log_reg_complete_cv.mean())*100,
                (tree_complete_train - tree_complete_cv.mean())*100,
                (rf_complete_train - rf_complete_cv.mean())*100,
                (knn_complete_train - knn_complete_cv.mean())*100]
})

print("COMPARACIÓN COMPLETA: BÁSICO vs COMPLETO")
comparison

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

basic_models = comparison[comparison['Estrategia'] == 'Básico']
complete_models = comparison[comparison['Estrategia'] == 'Completo']

x = np.arange(4)
width = 0.35

color_basic = '#5DADE2'
color_complete = '#F39C12'

axes[0].bar(x - width/2, basic_models['CV (%)'].values, width, 
            label='Básico (sin gammagrafía)', alpha=0.85, color=color_basic, edgecolor='#2874A6', linewidth=1.5)
axes[0].bar(x + width/2, complete_models['CV (%)'].values, width,
            label='Completo (con gammagrafía)', alpha=0.85, color=color_complete, edgecolor='#BA4A00', linewidth=1.5)
axes[0].set_ylabel('CV Accuracy (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Comparación: Modelo Básico vs Completo', fontweight='bold', fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels(['LogReg', 'Tree', 'RF', 'KNN'])
axes[0].axhline(y=dummy_test_acc*100, color='#E74C3C', linestyle='--', alpha=0.6, linewidth=2, 
                label=f'Baseline ({dummy_test_acc*100:.1f}%)')
axes[0].legend(framealpha=0.9, loc='upper right')
axes[0].set_ylim([40, 100])
axes[0].grid(axis='y', alpha=0.3, linestyle='--')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

all_models = comparison[comparison['Estrategia'] != 'N/A']
colors_gap = ['#27AE60' if gap < 10 else '#F39C12' if gap < 20 else '#E74C3C' for gap in all_models['Gap (%)']]

axes[1].scatter(range(len(all_models)), all_models['Gap (%)'], 
                c=colors_gap, s=200, alpha=0.7, edgecolors='black', linewidth=1.5)

axes[1].axhline(y=10, color='#F39C12', linestyle='--', alpha=0.5, linewidth=2, label='Moderado (10%)')
axes[1].axhline(y=20, color='#E74C3C', linestyle='--', alpha=0.5, linewidth=2, label='Alto (20%)')

axes[1].set_ylabel('Gap (Train - CV) %', fontsize=12, fontweight='bold')
axes[1].set_title('Gap como Indicador de Overfitting', fontweight='bold', fontsize=13)
axes[1].set_xticks(range(len(all_models)))
axes[1].set_xticklabels(all_models['Modelo'], rotation=45, ha='right')
axes[1].legend(framealpha=0.9)
axes[1].grid(axis='y', alpha=0.3, linestyle='--')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


**Observaciones:**

- Todos los modelos superan el baseline — el ML aporta valor real en este problema
- Los modelos Completos (con gammagrafía) superan a los Básicos en todos los algoritmos

**¿Qué nos dice esta comparación más allá de los números?**

Acabamos de **cuantificar el valor clínico de añadir un test diagnóstico**. La diferencia de accuracy entre estrategia Básica y Completa representa exactamente cuánto mejora la predicción al incluir la gammagrafía de perfusión en el proceso diagnóstico.

Esto es una herramienta poderosa: en la práctica clínica, una gammagrafía cuesta dinero, tiempo y exposición al paciente. Con este análisis podemos responder objetivamente a la pregunta "¿vale la pena hacer la gammagrafía antes de decidir el cateterismo?" — y la respuesta es sí, porque mejora de forma consistente y en todos los algoritmos la capacidad predictiva del modelo.

Random Forest muestra el mejor balance en ambas estrategias. La razón, como veremos en detalle al seleccionarlo, es el **bagging**: al promediar 100 árboles cancela la varianza individual de cada uno sin aumentar el bias, lo que se traduce en la mayor ganancia sobre el baseline de todos los modelos probados.


**A partir de este momento continuamos con el Random Forest, y la elección no es arbitraria.**

Mirando la tabla comparativa, hay tres razones concretas:

1. **Mejor CV accuracy en ambas estrategias.** El RF supera a Logistic Regression porque puede capturar relaciones no lineales entre variables clínicas — algo que la regresión logística no puede hacer por su supuesto de linealidad. Supera a KNN porque no depende de una métrica de distancia global que puede ser distorsionada por features irrelevantes.

2. **Gap Train-CV razonable.** Comparado con el Decision Tree individual, el RF muestra mucho menos overfitting. Eso es el **bagging** en acción: entrenar 100 árboles con muestras aleatorias distintas y promediar sus votos hace que los errores individuales se cancelen entre sí, reduciendo drásticamente la varianza sin sacrificar bias.

3. **Más robusto a variaciones.** Al no depender de una única partición de los datos (como hace el árbol individual), el RF es más estable frente a variaciones entre pacientes — especialmente importante en medicina, donde la heterogeneidad entre casos es alta.

Para problemas tabulares de tamaño mediano como este, Random Forest es generalmente la primera elección sólida antes de pasar a métodos más complejos como XGBoost o redes neuronales.


 ---

 # SECCIÓN 13: Métricas Clínicas - Evaluación en Test Set



 ## Métricas que importan en medicina:

 Como me habreis oido decir antes, la accuracy no lo es todo, puesto que si el coste monetario o humano de un FP y un FN no son parecidos, no nos ayuda a encontrar un buen compromiso, de la misma manera que también nos engaña si predecimos sobre enfermedades con muy poca prevalencia (nuestros datos estan muy desbalanceados)

 **Sensitivity (Recall):** De todos los enfermos, ¿cuántos detectamos?

 - En medicina: NO queremos perder casos de enfermedad (priorizar Sensitivity)



 **Specificity:** De todos los sanos, ¿cuántos identificamos correctamente?

 - Evita procedimientos innecesarios



 **PPV (Precision):** Si predecimos "enfermedad", ¿qué probabilidad de estar en lo cierto?



 **NPV:** Si predecimos "sano", ¿qué probabilidad de estar en lo cierto?



 Evaluamos SOLO EN TEST SET (una única vez)

In [ ]:
def clinical_metrics(y_true, y_pred, model_name):
    """Calcula métricas clínicas relevantes"""
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    npv = TN / (TN + FN) if (TN + FN) > 0 else 0
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    
    print(f"\nMÉTRICAS CLÍNICAS: {model_name}")
    print(f"{'='*70}")
    print(f"\nConfusion Matrix:")
    print(f"  TN: {TN} (sanos identificados correctamente)")
    print(f"  FP: {FP} (sanos clasificados como enfermos - cateterismo innecesario)")
    print(f"  FN: {FN} (enfermos clasificados como sanos - PELIGROSO)")
    print(f"  TP: {TP} (enfermos identificados correctamente)")
    
    print(f"\nMÉTRICAS:")
    print(f"  Accuracy: {accuracy*100:.1f}%")
    print(f"  Sensitivity (Recall): {sensitivity*100:.1f}% - Detecta {sensitivity*100:.0f}% de enfermos")
    print(f"  Specificity: {specificity*100:.1f}% - Identifica {specificity*100:.0f}% de sanos")
    print(f"  PPV (Precision): {ppv*100:.1f}% - Si predice enfermedad, {ppv*100:.0f}% correcto")
    print(f"  NPV: {npv*100:.1f}% - Si predice sano, {npv*100:.0f}% correcto")
    
    print(f"\nCASOS PERDIDOS: {FN} pacientes con enfermedad NO detectados")
    
    return {
        'Model': model_name,
        'Sensitivity': sensitivity * 100,
        'Specificity': specificity * 100,
        'PPV': ppv * 100,
        'NPV': npv * 100,
        'Accuracy': accuracy * 100,
        'FN': FN,
        'FP': FP
    }


In [ ]:
# Evaluar Random Forest Básico en Test
print("EVALUACIÓN FINAL EN TEST SET")

y_pred_rf_basic = rf_basic.predict(X_test_basic)
metrics_rf_basic = clinical_metrics(y_test, y_pred_rf_basic, "Random Forest (Básico)")


In [ ]:
# Evaluar Random Forest Completo en Test
y_pred_rf_complete = rf_complete.predict(X_test_scaled)
metrics_rf_complete = clinical_metrics(y_test, y_pred_rf_complete, "Random Forest (Completo)")


In [ ]:
# Comparación visual de métricas
metrics_comparison = pd.DataFrame([metrics_rf_basic, metrics_rf_complete])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics_to_plot = ['Sensitivity', 'Specificity', 'PPV', 'NPV']
x = np.arange(len(metrics_to_plot))
width = 0.35

basic_vals = [metrics_rf_basic[m] for m in metrics_to_plot]
complete_vals = [metrics_rf_complete[m] for m in metrics_to_plot]

color_basic = '#5DADE2'
color_complete = '#F39C12'

axes[0].bar(x - width/2, basic_vals, width, label='Básico', alpha=0.85, 
            color=color_basic, edgecolor='#2874A6', linewidth=1.5)
axes[0].bar(x + width/2, complete_vals, width, label='Completo', alpha=0.85, 
            color=color_complete, edgecolor='#BA4A00', linewidth=1.5)
axes[0].set_ylabel('Porcentaje (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Métricas Clínicas: Básico vs Completo (Random Forest)', fontweight='bold', fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_to_plot)
axes[0].legend(framealpha=0.9)
axes[0].set_ylim([0, 100])
axes[0].axhline(y=80, color='#27AE60', linestyle='--', alpha=0.4, linewidth=2)
axes[0].grid(axis='y', alpha=0.3, linestyle='--')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

error_types = ['Falsos Negativos\n(Enfermos perdidos)', 'Falsos Positivos\n(Cateterismos innecesarios)']
basic_errors = [metrics_rf_basic['FN'], metrics_rf_basic['FP']]
complete_errors = [metrics_rf_complete['FN'], metrics_rf_complete['FP']]

x2 = np.arange(len(error_types))

axes[1].hlines(y=x2 - 0.15, xmin=0, xmax=basic_errors, color=color_basic, alpha=0.7, linewidth=3)
axes[1].plot(basic_errors, x2 - 0.15, "o", markersize=12, color=color_basic, 
             alpha=0.85, label='Básico', markeredgecolor='#2874A6', markeredgewidth=1.5)

axes[1].hlines(y=x2 + 0.15, xmin=0, xmax=complete_errors, color=color_complete, alpha=0.7, linewidth=3)
axes[1].plot(complete_errors, x2 + 0.15, "o", markersize=12, color=color_complete, 
             alpha=0.85, label='Completo', markeredgecolor='#BA4A00', markeredgewidth=1.5)

axes[1].set_xlabel('Número de casos', fontsize=12, fontweight='bold')
axes[1].set_title('Tipos de Error: Básico vs Completo', fontweight='bold', fontsize=13)
axes[1].set_yticks(x2)
axes[1].set_yticklabels(error_types)
axes[1].legend(framealpha=0.9)
axes[1].grid(axis='x', alpha=0.3, linestyle='--')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


 ---

 # SECCIÓN 14: Curvas ROC y Análisis de Threshold



 ## Por qué importa el threshold



 Los modelos de clasificación devuelven una probabilidad de enfermedad (0 a 1).

 Necesitamos un umbral de decisión (threshold) para convertir esa probabilidad en una decisión binaria. Hasta ahora automatiamente el codigo habia usado un threshold de 0.5 para binarizar nuestra prediction (que es una probabilidad entre 0 y 1) a un resultado (sano/enfermo). Cuando hacemos:

 y_pred_rf_complete = rf_complete.predict(X_test_scaled)

 Internamente python esta cogiendo el modelo rf_complete, los datos X_test_scaled, y esta generando un valor de probabilidad de pertenecer a la clase 1 para cada paciente. Entonces para el paciente 1 tendriamos una probabilidad de 0.12, para el 2 de 0.89 etc. de estar enfermos (la clase 1 es enfermo). Justo despues de esto, la funcion predict aplica un threshold de decisión, para determinar si esa probabilidad de cada paciente es suficientemente alta como para decir que efectivamente esta enfermo. (De hecho lo que hace es df['predict_probability]> threshold para savar una variable binaria de True/false enfermo). El threshold que usa la funcion .predict por defecto es 0.5, pero nosotros somos más listos que la función y podemos modificar este threshold en función de nuestros intereses.

 **Threshold por defecto: 0.5**

 - Si probabilidad >= 0.5, predecir "enfermedad"

 - Si probabilidad < 0.5, predecir "sano"



 **Problema:** Este 0.5 es arbitrario y puede no ser óptimo para decisiones clínicas.



 **En medicina:**

 - **Threshold bajo (ej: 0.3):** Más sensible, detecta más enfermos, pero más falsos positivos

 - **Threshold alto (ej: 0.7):** Más específico, menos falsos positivos, pero pierde casos reales



 La **curva ROC** muestra el trade-off entre sensibilidad y especificidad para TODOS los thresholds posibles. 
 
 **Importante: Deberemos identificar el threshold que es más util para nuestras necesidades clínicas**

Alerta: El modelo ya lo habiamos entrenado antes y escogido en train, cogiendo el que tenia mejor accuracy en cross-validation en TRAIN. Ahora todo lo estamos comprobando en TEST.

In [ ]:
y_proba_rf_basic = rf_basic.predict_proba(X_test_basic)[:, 1]
y_proba_rf_complete = rf_complete.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# 3. Definir threshold
threshold = 0.5

# 4. Crear histograma
plt.figure(figsize=(10, 6))

# Histograma separado por clase real
plt.hist(y_proba_rf_basic[y_test == 0], bins=30, alpha=0.6, label='Clase 0 (real)', color='blue', edgecolor='black')
plt.hist(y_proba_rf_basic[y_test == 1], bins=30, alpha=0.6, label='Clase 1 (real)', color='red', edgecolor='black')

# Línea del threshold
plt.axvline(x=threshold, color='green', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')

# Anotaciones
plt.xlabel('Probabilidad predicha para clase 1', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.title('Distribución de Probabilidades Predichas con Threshold de Decisión', fontsize=14)
plt.legend()
plt.grid(axis='y', alpha=0.3)

# Mostrar regiones de decisión
plt.text(0.25, plt.ylim()[1]*0.9, 'Predicho: Clase 0', 
         ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='grey', alpha=0.5))
plt.text(0.75, plt.ylim()[1]*0.9, 'Predicho: Clase 1', 
         ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='grey', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Obtener probabilidades predichas
fpr_basic, tpr_basic, thresholds_basic = roc_curve(y_test, y_proba_rf_basic)
auc_basic = auc(fpr_basic, tpr_basic)

fpr_complete, tpr_complete, thresholds_complete = roc_curve(y_test, y_proba_rf_complete)
auc_complete = auc(fpr_complete, tpr_complete)

print(f"AUC Modelo Básico: {auc_basic:.3f}")
print(f"AUC Modelo Completo: {auc_complete:.3f}")


ROC-AUC es una metrica de performance threshold acnostic, es decir, a diferencia de la accuracy, senstivity etc., AUC no depende de que threshold escojamos, sinó que nos da una intuicion de lo bien que lo hace el modelo en general.

In [ ]:
# Visualización: Curvas ROC
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

color_basic = '#5DADE2'
color_complete = '#F39C12'

axes[0].plot(fpr_basic, tpr_basic, linewidth=3, color=color_basic, alpha=0.85,
             label=f'Básico (AUC = {auc_basic:.3f})')
axes[0].plot(fpr_complete, tpr_complete, linewidth=3, color=color_complete, alpha=0.85,
             label=f'Completo (AUC = {auc_complete:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.4, label='Aleatorio (AUC = 0.5)')

idx_basic_05 = np.argmin(np.abs(thresholds_basic - 0.5))
axes[0].scatter(fpr_basic[idx_basic_05], tpr_basic[idx_basic_05], 
                s=150, c=color_basic, edgecolors='black', linewidth=2, zorder=5,
                marker='o', label=f'Threshold=0.5 (Básico)')

idx_complete_05 = np.argmin(np.abs(thresholds_complete - 0.5))
axes[0].scatter(fpr_complete[idx_complete_05], tpr_complete[idx_complete_05], 
                s=150, c=color_complete, edgecolors='black', linewidth=2, zorder=5,
                marker='s', label=f'Threshold=0.5 (Completo)')

axes[0].set_xlabel('1 - Especificidad (Tasa Falsos Positivos)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sensibilidad (Tasa Verdaderos Positivos)', fontsize=12, fontweight='bold')
axes[0].set_title('Curvas ROC: Básico vs Completo', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', framealpha=0.95, fontsize=10)
axes[0].grid(alpha=0.3, linestyle='--')
axes[0].set_xlim([-0.02, 1.02])
axes[0].set_ylim([-0.02, 1.02])
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

specificity_complete = 1 - fpr_complete


youden_idx = np.argmax(tpr_complete - fpr_complete)
optimal_threshold = thresholds_complete[youden_idx]

axes[1].plot(thresholds_complete, tpr_complete, linewidth=3, color='#27AE60', 
             alpha=0.85, label='Sensibilidad')
axes[1].plot(thresholds_complete, specificity_complete, linewidth=3, color='#E74C3C', 
             alpha=0.85, label='Especificidad')

axes[1].axvline(optimal_threshold, color='gray', linestyle='--', linewidth=2, alpha=0.6)
axes[1].scatter(optimal_threshold, tpr_complete[youden_idx], s=150, c='#27AE60', 
                edgecolors='black', linewidth=2, zorder=5)
axes[1].scatter(optimal_threshold, specificity_complete[youden_idx], s=150, c='#E74C3C', 
                edgecolors='black', linewidth=2, zorder=5)
axes[1].text(optimal_threshold + 0.05, 0.5, f'Óptimo\n{optimal_threshold:.2f}', 
             fontsize=11, fontweight='bold', ha='left')

axes[1].axvline(0.5, color='black', linestyle=':', linewidth=2, alpha=0.5, label='Default (0.5)')

axes[1].set_xlabel('Threshold de decisión', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Tasa (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Trade-off Sensibilidad vs Especificidad\n(Modelo Completo)', 
                  fontsize=14, fontweight='bold')
axes[1].legend(loc='best', framealpha=0.95, fontsize=10)
axes[1].grid(alpha=0.3, linestyle='--')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\nThreshold óptimo (Youden's Index): {optimal_threshold:.3f}")
print(f"Sensibilidad en threshold óptimo: {tpr_complete[youden_idx]*100:.1f}%")
print(f"Especificidad en threshold óptimo: {specificity_complete[youden_idx]*100:.1f}%")


**Interpretación de las curvas ROC — ¿qué estamos viendo exactamente?**

**Gráfico izquierdo (Curvas ROC):**

- **Eje X (1 - Especificidad):** Tasa de Falsos Positivos — qué proporción de pacientes sanos estamos mandando erróneamente al cateterismo. Queremos este valor bajo.
- **Eje Y (Sensibilidad):** Tasa de Verdaderos Positivos — qué proporción de enfermos detectamos. Queremos este valor alto.
- Cada punto de la curva corresponde a un threshold distinto. Moverse hacia arriba-izquierda es mejor (alta sensibilidad con pocos falsos positivos).
- **Línea diagonal punteada:** modelo aleatorio (AUC = 0.5). Predice al azar, sin valor clínico.
- **Puntos marcados:** corresponden al threshold por defecto de 0.5. Puedes ver visualmente cómo cambia el equilibrio sensibilidad/especificidad si mueves ese punto.
- **AUC (Área Bajo la Curva):** resume en un solo número la capacidad discriminativa del modelo independientemente del threshold. AUC = 0.5 es azar; AUC = 1.0 es perfecto. En medicina, AUC > 0.8 se considera bueno.

**Gráfico derecho (Trade-off Sensibilidad vs Especificidad):**

- Muestra cómo varían sensibilidad (verde) y especificidad (rojo) para cada valor de threshold.
- El **punto de cruce** es el threshold de Youden (maximiza la suma de ambas métricas).
- La **línea punteada negra** es el threshold por defecto (0.5).
- En nuestro caso clínico, queremos movernos hacia thresholds **más bajos** que el default: perder algo de especificidad (más cateterismos innecesarios) a cambio de ganar sensibilidad (no dejar enfermos sin detectar). El análisis de la sección siguiente lo cuantifica.

**Interpretación de las curvas ROC:**
- AUC = 0.5: Modelo aleatorio (línea diagonal) — no aporta valor
- AUC = 1.0: Modelo perfecto — nunca se da en la realidad
- AUC > 0.8: Buena discriminación clínica
- Youden's Index = max(Sensibilidad + Especificidad - 1): threshold que balancea ambas métricas por igual, aunque en medicina con costes asimétricos (FN > FP) puede no ser el óptimo clínico


---

## SECCIÓN 14.1: Curva Precision-Recall — Cuando la ROC no basta

La curva ROC es útil, pero tiene una limitación importante: **puede ser engañosamente optimista en datasets desbalanceados** (cuando una clase es mucho más rara que la otra).

### ¿Por qué la ROC puede engañar?

La curva ROC usa en el eje X la tasa de Falsos Positivos = FP / (FP + TN). Cuando hay **muchos negativos reales** (muchos pacientes sanos), el denominador (FP + TN) es grande, así que la tasa de FP parece baja aunque en términos absolutos haya muchos FP. La curva ROC "no nota" que estamos generando muchos errores en la clase mayoritaria.

La **curva Precision-Recall** evita este problema porque no incluye los TN en ningún denominador:
- **Recall (eje X)** = TP / (TP + FN) → ¿Qué % de enfermos detectamos?
- **Precision (eje Y)** = TP / (TP + FP) → De todos a quienes predecimos "enfermo", ¿qué % lo son de verdad?

Ambas métricas se centran **solo en la clase positiva (los enfermos)**, que es exactamente lo que nos importa clínicamente.

### El baseline cambia

En la curva ROC, el modelo aleatorio tiene siempre AUC = 0.5 (diagonal). En la curva Precision-Recall, el baseline es la **prevalencia de la clase positiva** (~45% en nuestro dataset). Si tu Average Precision (AP) no supera claramente ese valor, el modelo no aporta valor real.

### Cuándo usar cada una

| Situación | Recomendación |
|---|---|
| Dataset equilibrado | ROC-AUC es suficiente |
| Dataset desbalanceado (ej. enfermedad rara, 1-5%) | **Usa la curva PR** |
| Te importa más no perder positivos | PR más informativa |
| Necesitas comparar modelos de forma general | Ambas |

**En nuestro caso:** el dataset está relativamente equilibrado (~45% positivos), así que ROC y PR cuentan historias similares. Pero en problemas reales de medicina (enfermedades raras) la diferencia es dramática — un modelo con ROC-AUC 0.85 puede tener AP de 0.3 si la clase positiva es muy escasa. Por eso conviene ver siempre ambas.


In [ ]:
# Calcular curvas Precision-Recall para ambos modelos
precision_basic, recall_basic, _ = precision_recall_curve(y_test, y_proba_rf_basic)
ap_basic = average_precision_score(y_test, y_proba_rf_basic)

precision_complete, recall_complete, _ = precision_recall_curve(y_test, y_proba_rf_complete)
ap_complete = average_precision_score(y_test, y_proba_rf_complete)

print(f"Average Precision (AP) — Modelo Básico:   {ap_basic:.3f}")
print(f"Average Precision (AP) — Modelo Completo: {ap_complete:.3f}")
print(f"\nComparación con ROC-AUC:")
print(f"  ROC-AUC Básico:   {auc_basic:.3f}  |  AP Básico:   {ap_basic:.3f}")
print(f"  ROC-AUC Completo: {auc_complete:.3f}  |  AP Completo: {ap_complete:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

color_basic = '#5DADE2'
color_complete = '#F39C12'
prevalencia = y_test.mean()  # Baseline de la curva PR

axes[0].plot(recall_basic, precision_basic, color=color_basic, linewidth=3,
             label=f'Básico (AP = {ap_basic:.3f})', alpha=0.85)
axes[0].plot(recall_complete, precision_complete, color=color_complete, linewidth=3,
             label=f'Completo (AP = {ap_complete:.3f})', alpha=0.85)
axes[0].axhline(y=prevalencia, color='#E74C3C', linestyle='--', linewidth=2, alpha=0.7,
                label=f'Baseline aleatorio ({prevalencia:.2f})')
axes[0].set_xlabel('Recall (Sensibilidad)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precision (PPV)', fontsize=12, fontweight='bold')
axes[0].set_title('Curvas Precision-Recall\nBásico vs Completo', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower left', framealpha=0.9)
axes[0].grid(alpha=0.3, linestyle='--')
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.05])
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

x = np.arange(2)
width = 0.35
bars1 = axes[1].bar(x - width/2, [auc_basic*100, auc_complete*100], width,
                     label='ROC-AUC (%)', color='#85C1E9', edgecolor='black', linewidth=1.2, alpha=0.9)
bars2 = axes[1].bar(x + width/2, [ap_basic*100, ap_complete*100], width,
                     label='Average Precision (%)', color='#F8C471', edgecolor='black', linewidth=1.2, alpha=0.9)
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 0.5, f'{h:.1f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(['Básico', 'Completo'], fontsize=12)
axes[1].set_ylabel('Valor (%)', fontsize=12, fontweight='bold')
axes[1].set_title('ROC-AUC vs Average Precision\n¿Cuentan la misma historia?', fontsize=13, fontweight='bold')
axes[1].legend(framealpha=0.9); axes[1].set_ylim([50, 100])
axes[1].grid(axis='y', alpha=0.3, linestyle='--')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


## Calibración del modelo: ¿Las probabilidades significan lo que creemos?

Hasta ahora hemos usado las probabilidades del modelo (output de `predict_proba`) para seleccionar thresholds. Pero hay una pregunta implícita que no hemos hecho: **¿son esas probabilidades fiables?**

Cuando el modelo dice "este paciente tiene un 70% de probabilidad de enfermedad", ¿significa que de 100 pacientes similares, ~70 tienen realmente la enfermedad?

Eso es lo que mide la **calibración**: la correspondencia entre las probabilidades predichas y las frecuencias reales observadas.

### Cómo interpretar el diagrama

En el **eje X** están las probabilidades que predice el modelo (agrupadas en bins). En el **eje Y** está la fracción real de positivos en cada bin.

- **Línea diagonal perfecta** (punteada): calibración perfecta. Si predice 0.7, el 70% de esos pacientes realmente están enfermos.
- **Curva por encima** de la diagonal: el modelo es **conservador** — subestima las probabilidades. Cuando dice 0.7, en realidad más del 70% están enfermos.
- **Curva por debajo** de la diagonal: el modelo es **sobreconfiado** — sobreestima. Cuando dice 0.7, en realidad menos del 70% están enfermos.

### ¿Por qué importa?

Si usamos las probabilidades del modelo para fijar un threshold (por ejemplo, 0.3 para capturar todos los positivos), ese threshold solo tiene significado clínico directo si el modelo está bien calibrado. Un modelo mal calibrado puede decir "probabilidad 0.3" cuando la frecuencia real de enfermedad en ese grupo es 0.6 — y nuestro threshold no estaría haciendo lo que creemos.

**Nota sobre Random Forests:** tienden a comprimir las probabilidades hacia valores intermedios (raramente predicen 0.05 o 0.95 con confianza), lo que las aleja de la diagonal. Esto es conocido y esperable.


In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_proba, model_name, color in zip(
    axes,
    [y_proba_rf_basic, y_proba_rf_complete],
    ['Modelo Básico (sin gammagrafía)', 'Modelo Completo (con gammagrafía)'],
    ['#5DADE2', '#F39C12']):

    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=8, strategy='quantile')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5, label='Calibración perfecta')
    ax.plot(prob_pred, prob_true, 'o-', color=color, linewidth=2.5, markersize=9,
            label=model_name, markeredgecolor='black', markeredgewidth=1.2)

    ax.set_xlabel('Probabilidad predicha por el modelo', fontsize=11, fontweight='bold')
    ax.set_ylabel('Fracción de positivos reales', fontsize=11, fontweight='bold')
    ax.set_title(f'Diagrama de Calibración\n{model_name}', fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.3, linestyle='--')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\nRecuerda: los Random Forests tienden a comprimir probabilidades hacia valores")
print("intermedios (alejándose de 0 y 1). Si la calibración no es perfecta, el threshold")
print("seleccionado puede no corresponder exactamente con la frecuencia real de enfermedad.")
print("Existen técnicas para corregirlo (Platt Scaling, Isotonic Regression), aunque")
print("quedan fuera del alcance de este script.")


In [ ]:
# Análisis de diferentes thresholds con el modelo básico
#
# Nota: roc_curve() ya devuelve internamente todos los thresholds y sus métricas.
# Hacemos este bucle manualmente para que veáis exactamente qué hay "dentro" de esa función
# y cómo se construyen las métricas punto a punto. En producción usaríais roc_curve() directamente.

thresholds_to_test = np.linspace(0, 1, num=51)
results = []

for threshold in thresholds_to_test:
    y_pred_threshold = (y_proba_rf_basic >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred_threshold)
    TN, FP, FN, TP = cm.ravel()

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    npv = TN / (TN + FN) if (TN + FN) > 0 else 0

    results.append({
        'Threshold': threshold,
        'Sensitivity': sensitivity * 100,
        'Specificity': specificity * 100,
        'Accuracy': (TP + TN) / (TP + TN + FP + FN) * 100,
        'PPV': ppv * 100,
        'NPV': npv * 100,
        'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
        'Total_Errors': FN + FP
    })

results_df = pd.DataFrame(results)
print("Análisis de diferentes thresholds (Modelo Básico):")
print(results_df.round(3).to_string(index=False))


 **Interpretación clínica de los thresholds:**

 Gracias a esta visualización, ahora podremos escoger el threshold que nos de unas metricas más utiles para nuestras necesidades.


 **Decisión clínica:**

 En nuestro caso, que no nos podemos permitir perder casos de enfermedad,

 un threshold alrededor de 0.16 podría ser el más apropiado, puesto que no estariamos pasando por alto ningun positivo, y estamos descartando un ~64% de pacientes sanos que ya no tendrán que hacerse el cateterismo.

 Evidentemente todo esto se tendria que testear en un entorno clínico real para ver si los resultados se mantienen. (En un dataset de validación externa)

 ## Victoria!

# SECCIÓN 15: Importancia de las variables (Si hay tiempo)



In [ ]:
# Feature Importance - RF Básico
importances_basic = rf_basic.feature_importances_
indices_basic = np.argsort(importances_basic)[::-1]

plt.figure(figsize=(10, 6))
plt.barh(range(len(importances_basic)), importances_basic[indices_basic], 
         color='steelblue', edgecolor='black')
plt.yticks(range(len(importances_basic)), 
           [basic_features[i] for i in indices_basic])
plt.xlabel('Importancia', fontsize=12)
plt.title('Feature Importance - Random Forest Básico', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()



In [ ]:
# Feature Importance - RF Completo
importances_complete = rf_complete.feature_importances_
indices_complete = np.argsort(importances_complete)[::-1]

plt.figure(figsize=(10, 6))
plt.barh(range(len(importances_complete)), importances_complete[indices_complete], 
         color='orange', edgecolor='black')
plt.yticks(range(len(importances_complete)), 
           [complete_features[i] for i in indices_complete])
plt.xlabel('Importancia', fontsize=12)
plt.title('Feature Importance - Random Forest Completo', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Interpretación Clínica del Feature Importance (Modelo Completo):

Las tres variables con mayor importancia son:

- thal (gammagrafía cardíaca): Aparece como la variable más determinante, lo que sugiere su relevancia diagnóstica en la detección de enfermedad coronaria y apoya su indicación en casos ambiguos tras prueba de esfuerzo no concluyente.
- thalach (frecuencia cardíaca máxima alcanzada): Indica la capacidad funcional del corazón durante el esfuerzo. Una frecuencia máxima reducida podría estar relacionada con limitación coronaria.
- oldpeak (depresión del segmento ST): Esta medida electrocardiográfica durante el esfuerzo tiende a reflejar isquemia miocárdica, donde valores elevados suelen asociarse con obstrucción coronaria significativa.

Otras variables como el tipo de dolor torácico (cp_4), edad y colesterol también muestran contribuciones notables. En contraste, variables como sexo y resultados de ECG en reposo presentan menor peso predictivo, aunque contribuyen al modelo global.

Este patrón de importancia parece coherente con el conocimiento clínico: las pruebas funcionales (esfuerzo, gammagrafía) y los signos de isquemia (como la depresión del ST o defectos de perfusión) tienden a ser más predictivos que los factores de riesgo basales (edad, sexo, colesterol) para identificar enfermedad coronaria establecida que requiere intervención.

# SECCIÓN 16: Entrenamiento final.

Ahora que ya tenemos el mejor modelo y el mejor threshold, podemos entrenarlo con todos los datos, guardarlo y ponerlo en producción!

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECCIÓN 16: Entrenamiento final
# ─────────────────────────────────────────────────────────────────────────────
#
# Ahora que tenemos el mejor modelo, el proceso correcto es:
#   1. Seleccionar el threshold USANDO SOLO datos de TRAIN (cross-validation)
#   2. Evaluar el impacto clínico en el test set (una sola vez)
#   3. Entrenar el modelo final con todos los datos (train + test)
#
# IMPORTANTE: El threshold NO puede seleccionarse usando el test set.
# Si lo hiciésemos, estaríamos ajustando una decisión de diseño a esos datos concretos
# de test — y el threshold no generalizaría a nuevos pacientes en producción.
# Es el mismo principio que no usar el test set para entrenar el modelo.

# PASO 1: Selección del threshold mediante Cross-Validation (SOLO en train)
# ──────────────────────────────────────────────────────────────────────────
cv_threshold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
thresholds_por_fold = []

print("Buscando threshold óptimo mediante Cross-Validation (solo datos de TRAIN)...")
print("="*70)

for fold, (train_idx, val_idx) in enumerate(cv_threshold.split(X_train_basic, y_train)):
    X_tr_fold = X_train_basic.iloc[train_idx]
    X_val_fold = X_train_basic.iloc[val_idx]
    y_tr_fold  = y_train.iloc[train_idx]
    y_val_fold = y_train.iloc[val_idx]

    modelo_fold = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
    modelo_fold.fit(X_tr_fold, y_tr_fold)

    y_proba_fold = modelo_fold.predict_proba(X_val_fold)[:, 1]
    fpr_f, tpr_f, thr_f = roc_curve(y_val_fold, y_proba_fold)

    # Threshold más alto que mantiene sensibilidad = 100%
    idx_100 = np.where(tpr_f >= 1.0)[0]
    th_fold = thr_f[idx_100[0]] if len(idx_100) > 0 else thr_f[np.argmax(tpr_f)]
    thresholds_por_fold.append(th_fold)

    sens_fold = tpr_f[idx_100[0]] if len(idx_100) > 0 else tpr_f[np.argmax(tpr_f)]
    print(f"  Fold {fold+1}: threshold = {th_fold:.3f}  → Sensibilidad en validación = {sens_fold*100:.0f}%")

# Usamos la mediana (más robusta que la media ante valores atípicos entre folds)
optimal_threshold_basic = float(np.median(thresholds_por_fold))
print(f"\nThreshold óptimo (mediana de 5 folds): {optimal_threshold_basic:.3f}")
print("Este threshold se estimó ÚNICAMENTE con datos de train — el test set sigue intacto.")

# PASO 2: Evaluar impacto clínico con el threshold seleccionado (en test)
# ───────────────────────────────────────────────────────────────────────
y_pred_rf_basic_optimal = (y_proba_rf_basic >= optimal_threshold_basic).astype(int)
cm_opt = confusion_matrix(y_test, y_pred_rf_basic_optimal)
TN_opt, FP_opt, FN_opt, TP_opt = cm_opt.ravel()

total_sanos = TN_opt + FP_opt
porcentaje_evitados = (TN_opt / total_sanos) * 100 if total_sanos > 0 else 0

print("\nIMPACTO CLÍNICO — MODELO BÁSICO CON THRESHOLD ÓPTIMO")
print("="*70)
print(f"Threshold aplicado: {optimal_threshold_basic:.3f}")
print(f"\nPacientes sin enfermedad (n={total_sanos}):")
print(f"  Cateterismos evitados:      {TN_opt} ({porcentaje_evitados:.1f}%)")
print(f"  Cateterismos innecesarios:  {FP_opt} ({(FP_opt/total_sanos*100) if total_sanos>0 else 0:.1f}%)")
print(f"\nPacientes con enfermedad (n={TP_opt + FN_opt}):")
print(f"  Correctamente detectados:   {TP_opt} ({(TP_opt/(TP_opt+FN_opt)*100) if (TP_opt+FN_opt)>0 else 0:.1f}%)")
print(f"  Casos no detectados (FN):   {FN_opt} ({(FN_opt/(TP_opt+FN_opt)*100) if (TP_opt+FN_opt)>0 else 0:.1f}%)")
print(f"\nBeneficio estimado:")
print(f"  Reducción de cateterismos en sanos: {porcentaje_evitados:.1f}%")
print(f"  Ahorro económico estimado: {TN_opt * 3000:,} EUR")
print(f"  Reducción de complicaciones: ~{int(TN_opt * 0.02)} pacientes")
print("="*70)

# PASO 3: Entrenar modelo final con TODOS los datos (train + test)
# ────────────────────────────────────────────────────────────────
# Ahora que el threshold ya está fijado (no usamos test para nada más),
# reentrenamos con todos los datos disponibles para maximizar la información.
X_all_basic = pd.concat([X_train_basic, X_test_basic], axis=0)
y_all       = pd.concat([y_train, y_test], axis=0)

rf_basic_final = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_basic_final.fit(X_all_basic, y_all)

import joblib, os
os.makedirs('./models', exist_ok=True)
joblib.dump(rf_basic_final, './models/rf_heart_basic_final.pkl')
joblib.dump(optimal_threshold_basic, './models/optimal_threshold_basic.pkl')

print(f"\nModelo final guardado. Threshold óptimo: {optimal_threshold_basic:.3f}")
print("Listo para producción (o validación externa).")


 ---

 # FIN DEL SCRIPT



 ## Conceptos Cubiertos:

 - Preparación de datos completa (missing values, encoding, scaling)

 - Identificación y manejo de data leakage (variable `ca`, pacientes repetidos)

 - **CRÍTICO: Imputación y escalado DESPUÉS del split para evitar data leakage**

 - Dos estrategias de modelado (Básico vs Completo)

 - Train/Test Split y Cross-Validation

 - Modelos baseline y 4 algoritmos de clasificación

 - Evaluación correcta (Train/Test/CV)

 - Métricas clínicas (Sensitivity, Specificity, PPV, NPV)

 - Curvas ROC y análisis de threshold

 - Impacto clínico real (ahorro económico, reducción de complicaciones)

 - Trade-offs clínicos (casos detectados vs procedimientos evitados)

 - Entrenar el modelo final y guardar para validación externa / Producción

---

## ¿Qué explorar a continuación?

Este script cubre el pipeline fundamental de ML supervisado para clasificación. Los siguientes pasos naturales son:

- **Gradient Boosting (XGBoost, LightGBM):** alternativas a Random Forest que en muchos problemas tabulares dan mejor accuracy con más eficiencia. Mismo paradigma de ensemble pero con árboles que se construyen secuencialmente (corrigiendo los errores del anterior).
- **Ajuste de hiperparámetros (GridSearchCV, RandomizedSearchCV):** buscar sistemáticamente los valores óptimos de `n_estimators`, `max_depth`, etc. en lugar de usar valores por defecto.
- **SHAP (SHapley Additive exPlanations):** explicar por qué el modelo toma cada decisión concreta para cada paciente — crucial en medicina para que el clínico entienda y confíe en la predicción.
- **Manejo de desbalanceo (SMOTE, class_weight):** cuando la clase positiva es muy minoritaria (ej. enfermedad rara), técnicas específicas para que el modelo no ignore esa clase.
- **Nested cross-validation:** evaluación estadísticamente imparcial cuando no se tiene suficiente datos para separar un test set independiente.
- **Calibración de probabilidades (Platt Scaling, Isotonic Regression):** corregir la tendencia de Random Forest a comprimir probabilidades, para que el threshold tenga un significado clínico más directo.
